# Práctica: de una red densa a una CNN con MNIST

**Extensión nueva de Clase 3.** Los cuadernos anteriores trabajan perceptrones multicapa; aquí introducimos convoluciones sin cambiar el problema ni los datos de comparación.

**Objetivo:** comparar dos hipótesis de representación, no demostrar que una CNN siempre gana. Un MLP conecta cada píxel con neuronas; una CNN comparte filtros locales. Menos parámetros no garantizan mayor precisión.

**Ruta orientativa (20 minutos de trabajo, más el cómputo necesario):** predecir y preparar (4 min), interpretar arquitecturas (4 min), entrenar y leer curvas (5 min), evaluar y discutir errores (7 min). La descarga y el entrenamiento dependen del equipo; no hay resultados precalculados.

Ejecutar en orden en Google Colab o un entorno con `tensorflow`, `scikit-learn`, `matplotlib` y `pandas`. La primera carga de MNIST requiere internet. No hacen falta widgets ni archivos privados.

## 1. Antes de ejecutar

1. ¿Qué pierde el MLP al aplanar una imagen: los valores de los píxeles o la estructura espacial explícita de su arquitectura?
2. ¿Por qué reutilizar un filtro podría necesitar menos parámetros?
3. Prediga qué modelo será más preciso y más rápido. Son hipótesis, no resultados.
4. Escriba un criterio antes de mirar prueba: comparar precisión, macro-F1, parámetros y tiempo; si dos modelos tienen precisión similar, ¿qué costo le importa?


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

SEED = 42
# Fijar estas decisiones ANTES de entrenar o consultar prueba.
TRAIN_LIMIT = 12000  # None utiliza las 54000 imágenes de entrenamiento disponibles.
MAX_EPOCHS = 8
BATCH_SIZE = 128
keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)


## 2. Un solo protocolo para ambos modelos

MNIST contiene dígitos manuscritos en escala de grises, no prendas Fashion-MNIST. Conservamos el conjunto oficial de prueba. Del conjunto oficial de entrenamiento separamos **una sola validación estratificada de 6000 imágenes**. Ambos modelos reciben exactamente los mismos índices.

La opción rápida selecciona 12000 imágenes de entrenamiento de manera estratificada; no reduce validación ni prueba. Si usa todos los datos, hágalo para ambos modelos y antes de evaluar prueba. Normalizar por 255 es una transformación fija que no estima estadísticas del conjunto de prueba.


In [ ]:
(images, labels), (test_images, test_labels) = keras.datasets.mnist.load_data()
all_indices = np.arange(len(images))
train_indices, val_indices = train_test_split(
    all_indices, test_size=6000, random_state=SEED, stratify=labels
)
if TRAIN_LIMIT is not None:
    if not 10 <= TRAIN_LIMIT <= len(train_indices):
        raise ValueError("TRAIN_LIMIT debe estar entre 10 y el tamaño de entrenamiento.")
    if TRAIN_LIMIT < len(train_indices):
        train_indices, _ = train_test_split(
            train_indices, train_size=TRAIN_LIMIT, random_state=SEED,
            stratify=labels[train_indices]
        )
assert not np.intersect1d(train_indices, val_indices).size
x_train = images[train_indices].astype("float32") / 255.0
x_val = images[val_indices].astype("float32") / 255.0
x_test = test_images.astype("float32") / 255.0
y_train, y_val, y_test = labels[train_indices], labels[val_indices], test_labels
# La CNN añade un eje de canal; los valores y el orden no cambian.
x_train_cnn, x_val_cnn, x_test_cnn = [x[..., None] for x in (x_train, x_val, x_test)]
print({"entrenamiento": len(y_train), "validación": len(y_val), "prueba": len(y_test)})
print("MLP:", x_train.shape, "CNN:", x_train_cnn.shape)
fig, axes = plt.subplots(1, 8, figsize=(10, 2))
for ax, image, label in zip(axes, x_train[:8], y_train[:8]):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Arquitectura y matemática mínima

**MLP de la práctica anterior:** `784 → 256 → 128 → 10`, con ReLU y Dropout(0.2). Una capa densa tiene `(entradas + 1) × salidas` parámetros, incluyendo sesgos: **235146** en total. Aplanar conserva los valores; no incorpora explícitamente su vecindad en la conectividad.

**CNN pequeña:** `28×28×1 → Conv(8) → 26×26×8 → Pool → 13×13×8 → Conv(16) → 11×11×16 → promedio global → 16 → 10`. Con kernel 3, stride 1, padding válido y dilatación 1: `salida = entrada − 3 + 1`. Cada capa convolucional tiene `(3×3×canales + 1) × filtros` parámetros: `80 + 1168 + 170 = 1418`.

El promedio global resume cada mapa con su media; favorece un modelo pequeño pero puede perder información espacial útil. Esta comparación cambia arquitectura **y capacidad**, por lo que no aísla causalmente el efecto de una sola capa.

Ambos terminan en softmax y usan `sparse_categorical_crossentropy`: nuestras etiquetas son enteros 0–9, no vectores one-hot. Para la clase verdadera, la pérdida es `−log(p)`: p=0.8 da aproximadamente 0.223; p=0.2 da 1.609.


In [ ]:
def build_mlp():
    return keras.Sequential([
        layers.Input(shape=(28, 28)), layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"), layers.Dropout(0.2),
        layers.Dense(10, activation="softmax")
    ], name="MLP")

def build_cnn():
    return keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(8, 3, activation="relu", padding="valid"),
        layers.MaxPooling2D(2),
        layers.Conv2D(16, 3, activation="relu", padding="valid"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation="softmax")
    ], name="CNN")

expected_params = {"MLP": (784 + 1)*256 + (256 + 1)*128 + (128 + 1)*10,
                   "CNN": (3*3*1 + 1)*8 + (3*3*8 + 1)*16 + (16 + 1)*10}
assert expected_params == {"MLP": 235146, "CNN": 1418}
models, histories, training_seconds = {}, {}, {}


## 4. Entrenar sin consultar prueba

Backpropagation calcula gradientes; Adam actualiza parámetros. Una época recorre el entrenamiento; un lote produce una actualización. EarlyStopping vigila **solo `val_loss`** y restaura los mejores pesos de validación.

La semilla se reinicia antes de construir cada modelo. Esto mejora la reproducibilidad, pero operaciones GPU y diferencias de versiones pueden producir variaciones. Mismo optimizador, límite de épocas, lote y datos no significan mismo tiempo ni mismo número efectivo de épocas.


In [ ]:
for name, builder, train_x, val_x in [
    ("MLP", build_mlp, x_train, x_val),
    ("CNN", build_cnn, x_train_cnn, x_val_cnn)
]:
    keras.utils.set_random_seed(SEED)
    model = builder()
    assert model.count_params() == expected_params[name]
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.summary()
    start = time.perf_counter()
    history = model.fit(
        train_x, y_train, validation_data=(val_x, y_val),
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, shuffle=True,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=2, restore_best_weights=True
        )], verbose=2
    )
    training_seconds[name] = time.perf_counter() - start
    models[name], histories[name] = model, history.history

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, history in histories.items():
    epochs = np.arange(1, len(history["loss"]) + 1)
    axes[0].plot(epochs, history["loss"], label=f"{name}: entrenamiento")
    axes[0].plot(epochs, history["val_loss"], "--", label=f"{name}: validación")
    axes[1].plot(epochs, history["val_accuracy"], label=name)
axes[0].set_title("Pérdida: ¿hay separación entre curvas?")
axes[1].set_title("Exactitud de validación")
for ax in axes:
    ax.set_xlabel("Época")
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


### Pausa de interpretación

- Si la pérdida de entrenamiento baja y la de validación sube, ¿qué evidencia sugiere sobreajuste?
- Si ambas siguen altas, ¿basta con afirmar que faltan épocas? Considere capacidad, tasa de aprendizaje y representación.
- Escriba su conclusión provisional **con validación**. Congele ahora arquitecturas e hiperparámetros. No utilice prueba para decidir nuevas variantes.

## 5. Evaluación final y errores

La siguiente celda llama a `evaluate` una vez por modelo. Después reutiliza predicciones para macro-F1, matriz de confusión y errores; esto es diagnóstico de la evaluación final, no ajuste. Si modifica modelos tras mirar estos resultados, este conjunto ya no es una prueba independiente para esa nueva decisión.

El tiempo mide `fit` (incluye validación y preparación inicial), no inferencia. Una sola ejecución en hardware compartido no establece una ventaja universal de velocidad.


In [ ]:
if "final_results" in globals():
    raise RuntimeError("La evaluación final ya se ejecutó. Reutilice final_results y predictions.")
rows, predictions = [], {}
for name, test_x in [("MLP", x_test), ("CNN", x_test_cnn)]:
    loss, accuracy = models[name].evaluate(test_x, y_test, verbose=0)
    predictions[name] = models[name].predict(test_x, verbose=0).argmax(axis=1)
    assert np.isclose(accuracy, accuracy_score(y_test, predictions[name]), atol=1e-6)
    rows.append({"modelo": name, "parámetros": models[name].count_params(),
                 "épocas": len(histories[name]["loss"]),
                 "segundos_fit": training_seconds[name], "pérdida_prueba": loss,
                 "accuracy_prueba": accuracy,
                 "macro_F1_prueba": f1_score(y_test, predictions[name], average="macro")})
final_results = pd.DataFrame(rows).set_index("modelo")
print(final_results.round(4).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, ["MLP", "CNN"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions[name], labels=np.arange(10), ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()

for name in ["MLP", "CNN"]:
    error_indices = np.flatnonzero(predictions[name] != y_test)[:8]
    if not len(error_indices):
        print(name, ": no hay errores en este conjunto.")
        continue
    fig, axes = plt.subplots(1, len(error_indices), figsize=(12, 2), squeeze=False)
    for ax, index in zip(axes.ravel(), error_indices):
        ax.imshow(x_test[index], cmap="gray")
        ax.set_title(f"Real {y_test[index]} / pred. {predictions[name][index]}")
        ax.axis("off")
    fig.suptitle(f"{name}: primeros errores, no muestra aleatoria")
    plt.tight_layout()
    plt.show()


## 6. Cierre: entregar evidencia, no solo un número

Complete: «Mi hipótesis inicial fue ____. En esta ejecución observé ____ de accuracy, ____ de macro-F1 y ____ segundos. La diferencia de parámetros es ____. Esto permite concluir ____, pero no demuestra ____».

1. Identifique dos confusiones entre dígitos. ¿La imagen es ambigua o propone una explicación verificable? No atribuya automáticamente significado humano a cada filtro.
2. ¿Compensa la reducción de parámetros si la CNN pierde precisión? Justifique según un presupuesto y una tarea concretos.
3. ¿Por qué una CNN no es automáticamente invariante a rotación o desplazamiento?
4. Para inspección de equipos, ¿por qué imágenes consecutivas del mismo activo no deben repartirse sin cuidado entre entrenamiento y prueba? Proponga separar por activo, fecha o fuente.
5. Un dígito por imagen es **clasificación**. ¿Qué etiquetas adicionales exigirían detección y segmentación?

**Límite de la práctica:** MNIST y una ejecución son una demostración didáctica. No validan desempeño industrial, equidad, robustez ni calibración. Una comparación más sólida necesita múltiples semillas y un protocolo definido antes de abrir una prueba independiente.

**Referencias y continuidad:** [Práctica ANN del repositorio](https://github.com/daprosero/Teoria-Aprendizaje-de-Maquina/blob/main/posgrado/Clase%203/Pr%C3%A1ctica_ANN.ipynb), [MNIST en Keras](https://keras.io/api/datasets/mnist/), [Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/), [CNN Explainer](https://poloclub.github.io/cnn-explainer/). CNN Explainer sirve para explorar operaciones e inferencia; este cuaderno sí entrena modelos.
